In [1]:
# ── Cell 1 : Imports ───────────────────────────────────────────────────────────
import json
import time
import datetime

import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from matplotlib.lines import Line2D

import liesel.model as lsl
import liesel.goose as gs
from tensorflow_probability.substrates.jax.experimental import distributions as tfde
from tensorflow_probability.python.internal.backend.jax.compat import v2 as tf
import tensorflow_probability.substrates.jax.distributions as tfd
import tensorflow_probability.substrates.jax.bijectors as tfb

import os 
import pickle

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (16, 9)

In [2]:
# ── Cell 2 : Data Loading Function ────────────────────────────────────────────
def load_margarine_data(filepath):
    """Loads and preprocesses the margarine dataset."""
    with open(filepath, 'r') as f:
        raw_data = json.load(f)
    
    X_list, y_list, Z_list, unit_indices = [], [], [], []
    
    n_alts_orig = 10
    kept_alts_1based = np.array([1, 2, 3, 4, 5, 7])
    kept_alts_0based = kept_alts_1based - 1 
    p = len(kept_alts_0based) 
    y_mapping = {val: idx for idx, val in enumerate(kept_alts_1based)}
    unit_idx_counter = 0
    
    for respondent in raw_data:
        X_raw = np.array(respondent['X'])
        y_raw = np.array(respondent['y'])
        
        if y_raw.ndim == 0:
            y_raw = np.atleast_1d(y_raw)
        
        n_choice_occasions_orig = len(y_raw)
        n_params_orig = X_raw.shape[1]
        X_reshaped = X_raw.reshape((n_choice_occasions_orig, n_alts_orig, n_params_orig))
        
        valid_occasions = np.isin(y_raw, kept_alts_1based)
        y_filtered = y_raw[valid_occasions]
        X_filtered = X_reshaped[valid_occasions]
        
        n_obs = len(y_filtered)
        if n_obs < 5:
            continue
            
        Z_raw = np.array(respondent.get('Z_raw', [1.0, 1.0]))
        Z_list.append(Z_raw)
        y_recoded = np.array([y_mapping[val] for val in y_filtered])
        X_new = np.zeros((n_obs, p, p))
        
        for r in range(n_obs):
            orig_prices = X_filtered[r, kept_alts_0based, 9]
            log_prices = np.log(orig_prices)
            X_new[r, 1:, 0:5] = np.eye(5)
            X_new[r, :, 5] = log_prices
            
        X_list.append(X_new)
        y_list.append(y_recoded)
        unit_indices.extend([unit_idx_counter] * n_obs)
        unit_idx_counter += 1

        Z_arr = np.array(Z_list)
        Z_arr[:, 0] = np.log(Z_arr[:, 0]) # Log Income

        # ── PER ROSSI: Center the data, but DO NOT add an intercept ──────────
        Z_arr[:, 0] -= np.mean(Z_arr[:, 0]) # De-mean Log Income
        Z_arr[:, 1] -= np.mean(Z_arr[:, 1]) # De-mean Family Size
        Z_final = Z_arr # No vector of ones appended

    return {
        "X": jnp.array(np.concatenate(X_list, axis=0)),
        "y": jnp.array(np.concatenate(y_list, axis=0)),
        "Z": jnp.array(Z_final),
        "unit_idx": jnp.array(unit_indices),
        "n_units": unit_idx_counter,
        "n_params": p,
        "n_z": Z_final.shape[1] # This will now correctly be 2 instead of 3
    }

In [3]:
# ── Cell 3 : Model Builder & Factories ────────────────────────────────────────
def make_wishart(df, scale_tril):
    return tfd.WishartTriL(
        df=df,
        scale_tril=scale_tril,
        input_output_cholesky=True,
        validate_args=False
    )

def make_mvn_precision(loc, precision_factor):
    return tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=loc,
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factor),
        validate_args=False
    )

def build_mixture_hmnl_model(
        data_dict,
        A_delta=0.01,
        a_mu=0.01,
        dirichlet_a=5.0
):
    """
    HMNL with mixture-of-normals heterogeneity.
    Prior structure matches bayesm::rhierMnlRwMixture.
    """
    n_params = int(data_dict["n_params"])
    n_units  = int(data_dict["n_units"])
    K_comp   = int(data_dict["K"])
    has_Z    = data_dict.get("Z") is not None

    # ── Wishart prior ─────────────────────────────────────────────────────────
    nu          = float(n_params + 3)
    V           = nu * jnp.eye(n_params)
    Vinv_chol   = jnp.linalg.cholesky(jnp.linalg.inv(V))
    Vinv_chol_K = jnp.broadcast_to(Vinv_chol[None], (K_comp, n_params, n_params))

    # ── pvec ~ Dirichlet ──────────────────────────────────────────────────────
    pvec = lsl.Var.new_param(
        value=jnp.ones(K_comp) / K_comp,
        distribution=lsl.Dist(tfd.Dirichlet,
                               concentration=jnp.ones(K_comp) * dirichlet_a),
        name="pvec"
    )
    pvec_latent = pvec.transform(tfb.SoftmaxCentered(), name="pvec_latent")

    # ── Sigma_k^{-1} ~ Wishart via Cholesky ──────────────────────────────────
    sigma_inv_chol_k = lsl.Var.new_param(
        value=jnp.broadcast_to(jnp.eye(n_params)[None], (K_comp, n_params, n_params)),
        distribution=lsl.Dist(
            make_wishart,
            df=jnp.full(K_comp, nu),
            scale_tril=Vinv_chol_K
        ),
        name="sigma_inv_chol_k"
    )
    sigma_inv_chol_k_latent = sigma_inv_chol_k.transform(
        tfb.FillScaleTriL(), name="sigma_inv_chol_k_latent"
    )

    # ── mu_k | Sigma_k ~ N(0, Sigma_k / a_mu) ────────────────────────────────
    mu_prec_factor_k = lsl.Var.new_calc(
        lambda L: jnp.sqrt(a_mu) * L,
        L=sigma_inv_chol_k,
        name="mu_prec_factor_k"
    )
    mu_k = lsl.Var.new_param(
        value=jnp.zeros((K_comp, n_params)),
        distribution=lsl.Dist(
            make_mvn_precision,
            loc=jnp.zeros(n_params),
            precision_factor=mu_prec_factor_k
        ),
        name="mu_k"
    )

    # ── Delta ~ N(0, (1/A_delta) * I) ────────────────────────────────────────
    if has_Z:
        n_demos           = int(data_dict["Z"].shape[1])
        Z_var             = lsl.Var.new_obs(data_dict["Z"], name="Z_obs")
        Delta_prec_factor = jnp.sqrt(A_delta) * jnp.eye(n_params)

        Delta = lsl.Var.new_param(
            value=jnp.zeros((n_demos, n_params)),
            distribution=lsl.Dist(
                make_mvn_precision,
                loc=jnp.zeros(n_params),
                precision_factor=Delta_prec_factor
            ),
            name="Delta"
        )
        z_delta = lsl.Var.new_calc(
            lambda z, d: z @ d, z=Z_var, d=Delta, name="z_delta"
        )

    # ── beta_i location: Z[i] @ Delta + mu_k  (n_units, K, n_params) ─────────
    if has_Z:
        beta_loc = lsl.Var.new_calc(
            lambda zd, mu: zd[:, None, :] + mu[None, :, :],
            zd=z_delta, mu=mu_k,
            name="beta_loc"
        )
    else:
        beta_loc = lsl.Var.new_calc(
            lambda mu: jnp.broadcast_to(mu[None, :, :], (n_units, K_comp, n_params)),
            mu=mu_k,
            name="beta_loc"
        )

    # Updated mixture function using precision factors
    def make_beta_mixture(pvec, locs, precision_factors):
        return tfd.MixtureSameFamily(
            mixture_distribution=tfd.Categorical(probs=pvec),
            components_distribution=tfde.MultivariateNormalPrecisionFactorLinearOperator(
                loc=locs,
                precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factors[None])
            )
        )

    # Updated beta_i definition
    beta_i = lsl.Var.new_param(
        value=jnp.zeros((n_units, n_params)),
        distribution=lsl.Dist(
            make_beta_mixture,
            pvec=pvec,
            locs=beta_loc,
            precision_factors=sigma_inv_chol_k  
        ),
        name="beta_i"
    )

    # ── Likelihood ────────────────────────────────────────────────────────────
    X_var         = lsl.Var.new_obs(data_dict["X"],        name="X_obs")
    idx_var       = lsl.Var.new_obs(data_dict["unit_idx"], name="idx_obs")
    beta_expanded = lsl.Var.new_calc(
        lambda b, idx: b[idx], b=beta_i, idx=idx_var, name="beta_expanded"
    )
    logits = lsl.Var.new_calc(
        lambda x, b: jnp.einsum("nij,nj->ni", x, b),
        x=X_var, b=beta_expanded,
        name="logits"
    )
    y_var = lsl.Var.new_obs(
        data_dict["y"],
        distribution=lsl.Dist(tfd.Categorical, logits=logits),
        name="y"
    )

    return lsl.Model([y_var])

In [4]:
# ── Cell 4 : Inference Engine ─────────────────────────────────────────────────
def run_mixture_inference(model, data_dict, chains=4, warmup=1000,
                          posterior=5000, seed=123):
    has_Z = data_dict.get("Z") is not None
    eb = gs.EngineBuilder(seed=seed, num_chains=chains)
    eb.set_model(gs.LieselInterface(model))
    eb.set_initial_values(model.state)

    # Block 1: Hyperparameters
    eb.add_kernel(gs.NUTSKernel(
        ["pvec_latent", "mu_k", "sigma_inv_chol_k_latent"],
        mm_diag=True 
    ))
    
    # Block 2: Covariates
    if has_Z:
        eb.add_kernel(gs.NUTSKernel(["Delta"]))

    # Block 3: Individual level parameters
    eb.add_kernel(gs.NUTSKernel(["beta_i"], mm_diag=True))

    eb.set_duration(warmup_duration=warmup, posterior_duration=posterior)

    print("Starting NUTS Sampling — Mixture HMNL...")
    engine = eb.build()
    engine.sample_all_epochs()
    return engine.get_results(), engine.get_results().get_posterior_samples()

In [5]:
# ── Cell 5 : Execution (Run Data, Build Model, Sample) ────────────────────────
print("Loading data...")
# Ensure "margarine_data.json" is in the same directory as the notebook
data_dict = load_margarine_data("margarine_data.json")

# Setup number of mixture components (K)
data_dict["K"] = 5
k_dim = int(data_dict["n_params"])

print(f"Data Loaded: {data_dict['n_units']} units, {k_dim} parameters, {data_dict['n_z']} demographic vars")
print(f"Total Observations: {data_dict['y'].shape[0]}")
print(f"Mixture Components (K): {data_dict['K']}")

print("Building 2-component mixture HMNL model...")
hmnl_model = build_mixture_hmnl_model(
    data_dict, A_delta=0.01, a_mu=0.01, dirichlet_a=5.0
)
print("Model built successfully.")

# Run inference
start_time = time.time()
n_warmup = 1000
n_posterior = 5000

mcmc_results, posterior_samples = run_mixture_inference(
    hmnl_model, data_dict, chains=4, warmup=n_warmup, posterior=n_posterior, seed=123
)

end_time = time.time()
formatted_time = str(datetime.timedelta(seconds=int(end_time - start_time)))
print(f"Sampling finished in {formatted_time} (H:M:S)")

Loading data...
Data Loaded: 313 units, 6 parameters, 2 demographic vars
Total Observations: 3405
Mixture Components (K): 5
Building 2-component mixture HMNL model...


c:\Users\ThinkPad\Desktop\Repositories\BDCM\liesel_project\.venv\Lib\site-packages\jax\_src\numpy\array_methods.py:122: UserWarning: Explicitly requested dtype float64 requested in astype is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


Model built successfully.


liesel.goose.builder - WARNING - No jitter functions provided. The initial values won't be jittered
liesel.goose.engine - INFO - Initializing kernels...


Starting NUTS Sampling — Mixture HMNL...


liesel.goose.engine - INFO - Done
liesel.goose.engine - INFO - Starting epoch: FAST_ADAPTATION, 75 transitions, 25 jitted together
100%|██████████████████████████████████████████| 3/3 [00:26<00:00,  8.94s/chunk]
liesel.goose.engine - WARNING - Errors per chain for kernel_00: 7, 51, 32, 43 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 0, 2, 2, 1 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 2, 2, 2, 2 / 75 transitions
liesel.goose.engine - INFO - Finished epoch
liesel.goose.engine - INFO - Starting epoch: SLOW_ADAPTATION, 25 transitions, 25 jitted together
100%|█████████████████████████████████████████| 1/1 [00:00<00:00, 493.10chunk/s]
liesel.goose.engine - WARNING - Errors per chain for kernel_00: 3, 19, 20, 21 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 1, 1, 2, 3 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 1, 1, 1, 1 / 25 transitions
liesel.g

Sampling finished in 2:12:27 (H:M:S)


In [6]:
import pickle

# Define the filename
filename = "margarine_5comp_results.pkl"

# Save both objects in a dictionary to the .pkl file
with open(filename, 'wb') as f:
    pickle.dump({
        'mcmc_results': mcmc_results, 
        'posterior_samples': posterior_samples
    }, f)

print(f"Results successfully saved to {filename}")

Results successfully saved to margarine_5comp_results.pkl
